# Business Intelligence — Tarea 4: Introducción a Series de Tiempo

**Hugo Roberto Tobías Mejía** · Facultad de Ingeniería, Universidad del Istmo de Guatemala

Este notebook corre la tarea completa de principio a fin a partir del archivo original
`data/Online Retail.xlsx`. El análisis escrito, con todas las respuestas a las preguntas
del enunciado, está en `informe/tarea4.pdf`; aquí se muestran los resultados en el orden
en que se producen.

**Nota sobre `statsmodels`.** El entorno donde se desarrolló esta tarea no tenía acceso a
PyPI, así que `statsmodels` no se pudo instalar y el modelo SARIMA está implementado a
mano en `src/utils_ts.py` (espacio de estados de Harvey + filtro de Kalman + máxima
verosimilitud exacta). La primera celda de código corre las 24 pruebas que verifican esa
implementación contra procesos simulados con parámetros conocidos.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

RAIZ = os.path.dirname(os.path.abspath("")) if os.path.basename(os.getcwd()) == "src" else os.getcwd()
sys.path.insert(0, os.path.join(RAIZ, "src"))
os.chdir(RAIZ)

import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

def ver(fig):
    display(Image(filename=os.path.join("figuras", fig)))

def tab(nombre, **kw):
    return pd.read_csv(os.path.join("tablas", nombre + ".csv"), **kw)

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

## Verificación de la implementación del SARIMA

Antes de usarla, se comprueba que la implementación recupera los parámetros de procesos
simulados cuyo valor verdadero se conoce.

In [ ]:
!python src/test_sarima.py

---
# Parte 0 — Procesos simulados

El inciso 0.1 es una derivación a mano y está en el PDF. Aquí van 0.2 a 0.5.

Se simulan $n=300$ pasos de un AR(1) con $c=0$ y $\sigma=1$, descartando los primeros 200,
y 5,000 réplicas por escenario con una semilla distinta en cada uno.

In [ ]:
import parte0
parte0.main()

In [ ]:
ver("p0_realizaciones.png")

**0.2 — La tabla de 5,000 réplicas.** Lo que hay que mirar: con $\phi=0$ la tasa de
rechazo es exactamente el 5 % nominal. Con $\phi=1$, $r_5$ vale 0.88 en vez del
$\phi^5=1$ teórico, porque esa fórmula supone estacionariedad. Y el estimado OLS de
$\phi$ tiene un sesgo hacia abajo que crece conforme $\phi \to 1$.

In [ ]:
tab("p0_tabla_ar1").round(4)

**0.4 — Las dos razones para reprobar.** Estos dos procesos **no tienen memoria**: cada
observación es independiente de las demás. Solo cambia la media con el tiempo.

La tendencia reprueba la prueba de rezago 1 el 98.5 % de las veces. El patrón semanal la
*pasa* (11.5 %) pese a tener un $r_7$ enorme: la prueba mira justo donde no hay nada.
Es el mismo fenómeno que RANDU en la Tarea 2.

In [ ]:
ver("p0_medias_moviles.png")
tab("p0_tabla_media_no_constante").round(4)

**0.5 — La prueba que sí ve.** Ljung–Box con $m=10$ junta los diez primeros rezagos en un
solo estadístico, así que el rezago 7 entra a la suma y el caso semanal se delata.

In [ ]:
tab("p0_tabla_ljungbox").round(4)

---
# Partes 1 y 2 — Limpieza y calendario

La variable a pronosticar es la **venta neta diaria**: la suma de `Quantity × UnitPrice` de
todas las líneas de un día, con las cancelaciones restando el día en que ocurren.

In [ ]:
import parte1_2
parte1_2.main()

**1.2 — Los seis casos.** Las dos decisiones de *conservar* son tan importantes como las
de eliminar: las cancelaciones deben restar el día en que ocurren, y las filas sin
`CustomerID` son £1.45 millones de venta real.

In [ ]:
pd.set_option("display.max_colwidth", 90)
tab("p1_casos")[["caso", "filas", "monto_GBP", "decision"]]

**1.3 — La bitácora, y por qué el orden importa.** La regla de cantidades negativas elimina
1,336 filas o 0 según cuándo se aplique, porque esas filas tienen todas precio cero y se
las lleva quien llegue primero. El conjunto final es idéntico; la bitácora no.

In [ ]:
display(tab("p1_bitacora"))
display(tab("p1_bitacora_orden_alterno"))

**1.4 — Las seis líneas más grandes.** Son tres pares venta/cancelación, no seis eventos.
Las seis sobreviven a la limpieza porque ninguna es un error; lo que las resuelve es netear
por día, ya que en los tres casos la cancelación ocurre el mismo día (12, 16 y 3 minutos
después).

In [ ]:
tab("p1_extremos")[["InvoiceNo", "Description", "Quantity", "monto_GBP",
                    "fecha_hora", "contraparte_mismo_dia"]]

**2.2 — El calendario.** De los 69 días sin transacciones, 53 son sábados: **la tienda no
abre los sábados**, pero sí abre los domingos. Esto es lo que fija $s=6$ en la Parte 4.
Los días restantes son feriados bancarios, tres de ellos móviles o irrepetibles.

In [ ]:
f = tab("p2_dias_sin_transacciones")
display(f[f.dia_semana != "sabado"])
ver("p2_serie.png")

---
# Parte 3 — Tendencia, estacionalidad y autocorrelación

Lo que hacen 3.1 y 3.2 es una descomposición clásica multiplicativa:
serie = tendencia × estacionalidad × resto.

In [ ]:
import parte3
parte3.main()

In [ ]:
ver("p3_tendencia.png")

**3.2 — Los índices estacionales.** Jueves y martes venden más del triple que el domingo.
El promedio de los seis índices es 1.2005 y no 1 porque el denominador promedia siete días
calendario, uno de los cuales es un sábado cerrado que entra como cero: lo subestima por un
factor de 6/7.

In [ ]:
ver("p3_indices.png")
tab("p3_indices_estacionales").round(4)

**3.3 — La ACF.** El pico está en el rezago 6, que son 7 días calendario: el mismo día de
la semana de la semana anterior. Los picos secundarios en 12 y 18 son dos y tres semanas
atrás. Esta es la evidencia empírica que justifica $s=6$.

In [ ]:
ver("p3_acf.png")
display(tab("p3_acf").round(4).T)
display(tab("p3_ljungbox"))

---
# Parte 4 — Del AR(1) al SARIMA

La receta: quitar la estacionalidad, mirar el residuo y modelar la memoria que quede.
Todo sobre los 245 días abiertos del entrenamiento, una fila por día.

In [ ]:
import parte4
resultados = parte4.main()

**4.1 — Quitar el día de la semana.** La ACF de los residuos decae *despacio*, con seis
rezagos consecutivos fuera de banda. Un AR(1) con $\phi \approx 0.28$ tendría
$r_3 \approx 0.02$ y ya estaría dentro de las bandas. Esa caída lenta es la pista visual
de una serie **no estacionaria**: al modelo le falta la tendencia.

In [ ]:
ver("p4_acf_residuos_ols.png")
tab("p4_ols_dow").round(2)

**4.2 — Un AR(1) no alcanza.** Limpia el rezago 1, pero deja fuera de banda los rezagos
2, 4, 6 y 9. El rezago 6 es estacionalidad, y un AR(1) con $\hat\phi = 0.28$ predice
$\rho_6 = 0.28^6 \approx 0.0005$: dice que ahí no hay nada, y sí lo hay.

In [ ]:
ver("p4_acf_residuos_ar1.png")
tab("p4_ar1").round(4)

**4.3 y 4.4 — Los dos SARIMA.** Ajustados sobre el **logaritmo** de la venta, con $s=6$.

El diagnóstico es el resultado central de esta parte: **S2 es el único que deja residuos de
ruido blanco**. S1 es estacionario y supone una media fija que no existe, así que la
tendencia de agosto en adelante se le queda en los residuos.

In [ ]:
display(tab("p4_sarima").round(4))
display(tab("p4_diagnostico").round(4))

**4.5 — Hacia dónde van los pronósticos.** S1 es estacionario y converge a su media;
S2 no lo es y se va sin límite, arrastrando el nivel y la pendiente del final de los datos.
La ventaja de S2 es una ventaja de **horizonte corto**.

In [ ]:
for clave, res in resultados.items():
    f = np.exp(res.forecast(120))
    print(f"{clave}: h=1 -> {f[0]:>10,.0f} GBP | h=30 -> {f[29]:>10,.0f} | h=120 -> {f[119]:>10,.0f}")

---
# Parte 5 — Pronóstico y fuga

Siete modelos evaluados de tres formas. El origen es siempre un domingo cuya venta ya
está observada, y los objetivos son los días abiertos de la semana siguiente.

In [ ]:
import parte5
parte5.main()

**5.2–5.3 — La tabla única.** (c) es la columna honesta, y **S2 gana por los dos
criterios**. (a) no aplica a B2, S1 y S2 porque los tres dependen del orden del tiempo.

Y los tres estimados hechos *dentro* del entrenamiento se quedan cortos: para M2, (b)
subestima el error real en un 39 %.

In [ ]:
r = tab("p5_resultados")
display(r.pivot_table(index="modelo", columns="estimacion", values="MAE").round(0))
display(r.pivot_table(index="modelo", columns="estimacion", values="RMSE").round(0))
ver("p5_prueba.png")

**5.4 — La trampa.** Agregar la media móvil **centrada** de 7 días como predictor mejora el
MAE un 9.8 % en (a) y un 9.4 % en (b), y **ninguna de las dos validaciones protesta**.

El predictor es ilegal porque mira al futuro: $\mathrm{MA}_t$ necesita $y_{t+1}$,
$y_{t+2}$ e $y_{t+3}$, y además contiene $y_t/7$, que es la propia respuesta. El origen
móvil no lo detecta porque la fuga viaja *dentro* de una columna calculada antes del corte.

La única señal de alarma es que (c) **no se puede calcular sin hacer trampa**.

In [ ]:
display(tab("p5_trampa"))
ver("p5_trampa.png")

---
# Parte 6 — Detrás de la serie hay clientes: análisis RFM

Todo se calcula usando **solo** las transacciones anteriores al 1 de octubre de 2011.
Mirar la ventana que se quiere predecir sería exactamente la fuga de la Parte 5.

In [ ]:
import parte6
parte6.main()

**6.2 — El problema con F.** F toma solo 48 valores distintos entre 3,658 clientes y 1,444
de ellos tienen $F=1$, casi dos quintiles enteros. No hay ningún corte que parta a los
clientes en cinco grupos iguales, así que clientes **idénticos** terminan en puntajes
distintos por un lanzamiento de moneda.

In [ ]:
rfm = pd.read_csv("datos_derivados/rfm.csv", index_col=0)
display(rfm.F.value_counts().sort_index().head(8).to_frame("clientes").T)
display(rfm.groupby("F_score").F.agg(["min", "max", "count"]))

**6.3 — Qué dimensión separa mejor.** **F**, por las dos medidas. R satura en el extremo
superior pero funciona muy bien en el inferior: R sirve para descartar, F para seleccionar.
La suma de los tres puntajes le gana a las tres por separado, que es la justificación
empírica de usar tres números y no uno.

In [ ]:
display(tab("p6_fraccion_por_puntaje").pivot(index="puntaje", columns="dimension",
                                             values="fraccion_compro").round(4))
display(tab("p6_auc").round(4))
ver("p6_roc.png")

**6.4 y 6.5 — Campeones y clientes nuevos.** El 21.4 % de los clientes genera el 53.5 % de
la venta del trimestre: casi exactamente la ley de Pareto, y el argumento comercial entero
del RFM en una línea.

Pero el 15.7 % de la venta vino de **723 clientes que el 1 de octubre no existían en la
base**. Ningún modelo cliente por cliente puede verlos, porque no tienen fila. Solo el
pronóstico agregado de la Parte 5 los incluye, sin saberlo. En el vocabulario de FPP2, la
Parte 5 es *top-down* y el RFM es *bottom-up*.

In [ ]:
display(tab("p6_campeones").T)
display(tab("p6_nuevos").T)

---
## Conclusiones

1. **La prueba que uno escoge decide lo que uno ve.** El proceso semanal no tiene memoria y
   pasa la prueba de rezago 1 el 88.5 % de las veces; Ljung–Box lo rechaza el 95.5 %.
2. **Reprobar la prueba de ruido blanco casi nunca significa "hay memoria".** Aquí venía de
   una media no constante: el día de la semana y la subida navideña.
3. **La limpieza es una decisión, no un procedimiento.** La decisión que más importa es una
   que *no* elimina: conservar el código `Manual`.
4. **El calendario es parte del modelo.** Que la tienda cierre los sábados es lo que fija
   $s=6$ y lo que explica el pico de la ACF en el rezago 6.
5. **El diagnóstico de residuos predijo al ganador.** S2 fue el único que pasó Ljung–Box en
   4.4 y el que ganó en prueba en 5.3; en ambos casos por la diferenciación.
6. **Una validación solo protege contra la fuga que ocurre después de ella.**
7. **Hay un error que ninguna validación puede estimar:** noviembre no se parece a mayo, y
   con un solo año de datos la estacionalidad anual ni siquiera está identificada.
8. **Detrás de la serie hay clientes, pero la serie sabe algo que los clientes no:** los
   clientes nuevos.

El análisis completo está en `informe/tarea4.pdf`.